# Auditing a clinical record extract for data-entry errors

`ov.tl.audit_clinical_tables` flags the **rows** of a multi-table clinical
extract (stays, labs, treatments, ...) whose recorded value is wrong for that
patient. Every reference is learned from the data. Rule families follow the
conformance / plausibility categories of Kahn et al. 2016 (*eGEMs* 4:1244); the
after-death rules are the data-derived counterpart of the OHDSI Data Quality
Dashboard `plausibleDuringLife` / `plausibleBeforeDeath` checks.

This tutorial uses a **synthetic ICU cohort** with known injected errors in two
layouts: absolute timestamps with a date of death, and **minute offsets** from
unit admission with death recorded as a discharge *status* ("Expired") plus the
offset of that discharge. Ids are nested: person > hospital admission > ICU stay.

In [1]:
import numpy as np, pandas as pd
import omicverse as ov


## 1. A synthetic cohort with known errors

The generator below writes the tables and returns `truth`: for each injected error kind, the set of `(table, row_id)` that carry it.

In [2]:
ETH = ["white", "black", "asian", "hispanic", "other"]
LABS = {"hgb": (12.0, 1.5, "g/dL"), "na": (140.0, 3.0, "mmol/L"), "cr": (None, 0.3, "mg/dL")}


def make_icu_cohort(seed=0, layout="offset", n_person=200, inject=True):
    """Return ``(tables, truth)``; ``truth`` maps an error category to a set of (table, row_id)."""
    rng = np.random.default_rng(seed)
    persons = []
    for i in range(n_person):
        sex = "M" if rng.random() < 0.5 else "F"
        persons.append(dict(pid=f"{1000 + i}", sex=sex, eth=rng.choice(ETH, p=[.6, .15, .1, .1, .05]),
                            height=rng.normal(176, 7) if sex == "M" else rng.normal(163, 6),
                            w0=float(np.exp(rng.normal(np.log(80), 0.18))),
                            birth=2100 - int(rng.integers(20, 85)) - float(rng.random()),
                            n_adm=int(rng.choice([1, 2, 3], p=[.5, .32, .18])), died=rng.random() < 0.15))
    stays, labs, treats = [], [], []
    k_stay = k_adm = k_lab = 0
    for p in persons:
        yrs = np.sort(rng.choice(np.arange(10), p["n_adm"], replace=False))
        for a_i, y in enumerate(yrs):
            k_adm += 1
            adm_id = f"{5000 + k_adm}"
            n_st = int(rng.choice([1, 2], p=[.75, .25]))
            adm_start = pd.Timestamp(f"{2100 + int(y)}-01-01") + pd.Timedelta(days=int(rng.integers(0, 300)))
            t_cursor = adm_start + pd.Timedelta(minutes=int(rng.integers(60, 3000)))
            for s_i in range(n_st):
                k_stay += 1
                last = p["died"] and a_i == len(yrs) - 1 and s_i == n_st - 1
                los = int(rng.integers(1000, 8000))
                hadm_off = -int((t_cursor - adm_start).total_seconds() // 60)
                hdis_off = los if last else los + int(rng.integers(500, 5000))
                age_y = (t_cursor - pd.Timestamp("2100-01-01")).days / 365.25 + 2100 - p["birth"]
                wa = p["w0"] * float(np.exp(rng.normal(0, 0.04)))
                st = dict(stay=f"{200000 + k_stay}", adm=adm_id, pid=p["pid"], sex=p["sex"], eth=p["eth"],
                          age=int(age_y), height=round(p["height"] + rng.normal(0, 0.8), 1), wa=round(wa, 1),
                          wd=round(wa * float(np.exp(rng.normal(0, 0.02))), 1), hadm_off=hadm_off, udis_off=los,
                          ustatus="Expired" if last else "Alive", hdis_off=hdis_off,
                          hstatus="Expired" if last else "Alive", year=adm_start.year, intime=t_cursor,
                          adm_start=adm_start, last=last)
                stays.append(st)
                # distinct draw instants per stay (two draws of one item at one minute would be a
                # genuine duplicate conflict)
                for off in rng.choice(np.arange(max(hadm_off, -600), hdis_off), 12, replace=False):
                    k_lab += 1
                    it = rng.choice(list(LABS))
                    mu, sd, u = LABS[it]
                    v = round(float(np.exp(rng.normal(0, sd))) if mu is None else float(rng.normal(mu, sd)), 2)
                    off = int(off)
                    labs.append(dict(lid=f"{k_lab}", stay=st["stay"], adm=adm_id, pid=p["pid"], off=off,
                                     item=it, v=v, u=u, t=t_cursor + pd.Timedelta(minutes=off)))
                for _ in range(3):
                    off = int(rng.integers(0, los))
                    treats.append(dict(stay=st["stay"], off=off, name=rng.choice(["abx", "vent", "pressor"])))
                t_cursor += pd.Timedelta(minutes=hdis_off + int(rng.integers(100, 2000)))
    S, L = pd.DataFrame(stays), pd.DataFrame(labs)
    truth = {k: set() for k in ("after_death", "demographic_change", "age_shift", "height_unit",
                                "weight_lb", "duplicate")}

    if layout == "offset":
        T = {"unit_stays": pd.DataFrame({
                "_row_id": "S" + S.index.astype(str), "unit_stay_id": S.stay, "hosp_adm_id": S.adm,
                "person_id": S.pid, "gender": S.sex.map({"M": "Male", "F": "Female"}), "age": S.age.astype(str),
                "ethnicity": S.eth, "height_cm": S.height.astype(str), "admit_weight": S.wa.astype(str),
                "discharge_weight": S.wd.astype(str), "hosp_admit_offset": S.hadm_off.astype(str),
                "unit_discharge_offset": S.udis_off.astype(str), "unit_discharge_status": S.ustatus,
                "hosp_discharge_offset": S.hdis_off.astype(str), "hosp_discharge_status": S.hstatus,
                "discharge_year": S.year.astype(str)}),
             "lab": pd.DataFrame({"_row_id": "L" + L.index.astype(str), "lab_id": L.lid, "unit_stay_id": L.stay,
                                  "lab_result_offset": L.off.astype(str), "lab_name": L["item"],
                                  "lab_result": L.v.astype(str)}),
             "treatment": pd.DataFrame({"_row_id": [f"T{i}" for i in range(len(treats))],
                                        "treatment_id": [f"{9000 + i}" for i in range(len(treats))],
                                        "unit_stay_id": [t["stay"] for t in treats],
                                        "treatment_offset": [str(t["off"]) for t in treats],
                                        "treatment_name": [t["name"] for t in treats]})}
        stay_t, eth_c, age_c, h_c, w_c, lab_t = "unit_stays", "ethnicity", "age", "height_cm", "admit_weight", "lab"
        multi_stay = S.pid.map(S.pid.value_counts()) >= 2
    else:
        P = S.groupby("pid").agg(sex=("sex", "first"), age=("age", "min")).reset_index()
        dod = S[S["last"]].set_index("pid").apply(lambda r: r.intime + pd.Timedelta(minutes=r.hdis_off), axis=1)
        A = S.groupby("adm").agg(pid=("pid", "first"), eth=("eth", "first"), start=("adm_start", "first"),
                                 end=("intime", "max")).reset_index()
        T = {"patients": pd.DataFrame({"_row_id": "P" + P.index.astype(str), "subject_id": P.pid, "gender": P.sex,
                                       "anchor_age": P.age.astype(str),
                                       "dod": P.pid.map(dod).map(lambda x: "" if pd.isna(x) else str(x.date()))}),
             "admissions": pd.DataFrame({"_row_id": "A" + A.index.astype(str), "hadm_id": A.adm, "subject_id": A.pid,
                                         "admittime": A.start.astype(str),
                                         "dischtime": (A.end + pd.Timedelta(days=12)).astype(str), "race": A.eth}),
             "icustays": pd.DataFrame({"_row_id": "I" + S.index.astype(str), "stay_id": S.stay, "hadm_id": S.adm,
                                       "subject_id": S.pid, "intime": S.intime.astype(str),
                                       "outtime": (S.intime + pd.to_timedelta(S.udis_off, unit="m")).astype(str),
                                       "admission_age": S.age.astype(str), "height": S.height.astype(str),
                                       "weight_admit": S.wa.astype(str), "weight_discharge": S.wd.astype(str)}),
             "labevents": pd.DataFrame({"_row_id": "L" + L.index.astype(str), "subject_id": L.pid, "hadm_id": L.adm,
                                        "stay_id": L.stay, "itemid": L["item"], "charttime": L.t.astype(str),
                                        "valuenum": L.v.astype(str), "valueuom": L.u}),
             "d_labitems": pd.DataFrame({"_row_id": ["D0", "D1", "D2"], "itemid": list(LABS),
                                         "label": ["hemoglobin", "sodium", "creatinine"]})}
        stay_t, eth_c, age_c, h_c, w_c, lab_t = "icustays", "race", "admission_age", "height", "weight_admit", "labevents"
        multi_stay = S.pid.map(S.pid.value_counts()) >= 2
    if not inject:
        return T, truth

    ST, LB = T[stay_t], T[lab_t]
    # 1. events after death
    dead_stays = set(S.stay[S["last"]])
    cand = L.index[L.stay.isin(dead_stays)]
    for i in rng.choice(cand, 3, replace=False):
        st = S[S.stay == L.loc[i, "stay"]].iloc[0]
        if layout == "offset":
            LB.loc[i, "lab_result_offset"] = str(st.hdis_off + int(rng.integers(600, 4000)))
        else:
            LB.loc[i, "charttime"] = str(st.intime + pd.Timedelta(minutes=int(st.hdis_off))
                                         + pd.Timedelta(days=int(rng.integers(2, 10))))
        truth["after_death"].add((lab_t, LB.loc[i, "_row_id"]))
    # 2. demographic attributes changing between stays / admissions of one person
    if layout == "offset":
        idx = rng.choice(ST.index[multi_stay.values], 5, replace=False)
        for j, i in enumerate(idx):
            if j < 3:
                ST.loc[i, eth_c] = rng.choice([e for e in ETH if e != ST.loc[i, eth_c]])
            else:
                ST.loc[i, "gender"] = "Female" if ST.loc[i, "gender"] == "Male" else "Male"
            truth["demographic_change"].add((stay_t, ST.loc[i, "_row_id"]))
    else:
        AD = T["admissions"]
        multi_adm = AD.subject_id.map(AD.subject_id.value_counts()) >= 2
        for i in rng.choice(AD.index[multi_adm.values], 3, replace=False):
            AD.loc[i, eth_c] = rng.choice([e for e in ETH if e != AD.loc[i, eth_c]])
            truth["demographic_change"].add(("admissions", AD.loc[i, "_row_id"]))
    pool = list(rng.permutation(ST.index[multi_stay.values]))
    # 3. age inconsistent with the person's other stays
    for i in pool[:2]:
        ST.loc[i, age_c] = str(int(ST.loc[i, age_c]) + int(rng.choice([-1, 1]) * rng.integers(12, 25)))
        truth["age_shift"].add((stay_t, ST.loc[i, "_row_id"]))
    # 4. height recorded in inches, 5. admission weight recorded in pounds
    rest = [i for i in rng.permutation(ST.index) if i not in pool[:2]]
    for i in rest[:2]:
        ST.loc[i, h_c] = str(round(float(ST.loc[i, h_c]) / 2.54, 1))
        truth["height_unit"].add((stay_t, ST.loc[i, "_row_id"]))
    for i in rest[2:5]:
        ST.loc[i, w_c] = str(round(float(ST.loc[i, w_c]) * 2.2046, 1))
        truth["weight_lb"].add((stay_t, ST.loc[i, "_row_id"]))
    # 6. exact duplicate event rows (fresh row id and fresh table-local surrogate id)
    dup = LB.loc[rng.choice(LB.index, 5, replace=False)].copy()
    dup["_row_id"] = [f"DUP{i}" for i in range(len(dup))]
    if "lab_id" in dup:
        dup["lab_id"] = [f"{900000 + i}" for i in range(len(dup))]
    T[lab_t] = pd.concat([LB, dup], ignore_index=True)
    truth["duplicate"] |= {(lab_t, r) for r in dup["_row_id"]}
    return T, truth

def score(findings, truth):
    fl = set(zip(findings['table'], findings['row_id']))
    allt = set().union(*truth.values())
    rec = {k: (len(v & fl) / len(v) if v else float('nan')) for k, v in truth.items()}
    return rec, (len(fl & allt) / len(fl) if fl else float('nan')), len(fl)

T, truth = make_icu_cohort(seed=1, layout='offset')
for k, v in T.items():
    print(f'{k:12s} {v.shape}  columns: {list(v.columns)[:9]}')
print({k: len(v) for k, v in truth.items()})

unit_stays   (418, 16)  columns: ['_row_id', 'unit_stay_id', 'hosp_adm_id', 'person_id', 'gender', 'age', 'ethnicity', 'height_cm', 'admit_weight']
lab          (5021, 6)  columns: ['_row_id', 'lab_id', 'unit_stay_id', 'lab_result_offset', 'lab_name', 'lab_result']
treatment    (1254, 5)  columns: ['_row_id', 'treatment_id', 'unit_stay_id', 'treatment_offset', 'treatment_name']
{'after_death': 3, 'demographic_change': 5, 'age_shift': 2, 'height_unit': 2, 'weight_lb': 3, 'duplicate': 5}


## 2. What the schema inference finds

The id hierarchy is discovered from the data (an id whose values each map to exactly one value of a coarser id), event times are recognised as offsets, and the death status column is paired with its offset.

In [3]:
schema = ov.tl.infer_clinical_schema(T)
print('id hierarchy (fine -> coarse):', schema['id_hierarchy'], '| person key:', schema['person'])
print('death encodings:', schema['demographics']['death_status'])
print({k: (v['kind'], v['time']) for k, v in schema['events'].items()})

id hierarchy (fine -> coarse): ['unit_stay_id', 'hosp_adm_id', 'person_id'] | person key: person_id
death encodings: [{'status': 'unit_discharge_status', 'dead_values': ['expired'], 'time': 'unit_discharge_offset', 'kind': 'offset'}, {'status': 'hosp_discharge_status', 'dead_values': ['expired'], 'time': 'hosp_discharge_offset', 'kind': 'offset'}]
{'lab': ('offset', 'lab_result_offset'), 'treatment': ('offset', 'treatment_offset')}


## 3. Audit and score against the injected truth

Each finding carries `family`, `rule`, an incident `cluster`, `severity` (robust z; empty for logical rules), `rule_rate` (share of the table's rows that rule flagged), `rule_rank` and `confidence`: `high` when the finding has a second signature (a logical contradiction, a decimal-shift / sign / unit-factor explanation, or an independent rule on the same row), `screen` for a graded deviation only.

In [4]:
f = ov.tl.audit_clinical_tables(T, external_reference=False, verbose=False)
print(f.groupby(['family', 'rule', 'confidence']).size().rename('rows').to_string())
rec, prec, n = score(f, truth)
print('recall per injected kind:', {k: round(v, 2) for k, v in rec.items()})
print(f'precision {prec:.2f} over {n} flagged rows')

family                rule             confidence
demographic_conflict  person_constant  high           6
                      person_numeric   high           6
                                       screen         3
duplicate             exact_duplicate  high           5
impossible_value      after_death      high           3
                      unit_factor      high          12
inconsistency         outside_stay     high           3
recall per injected kind: {'after_death': 1.0, 'demographic_change': 1.0, 'age_shift': 1.0, 'height_unit': 1.0, 'weight_lb': 1.0, 'duplicate': 1.0}
precision 0.87 over 23 flagged rows


## 4. The same cohort with absolute dates

The date layout (date of death, admission / discharge timestamps) goes through the same rules; nothing in the call changes.

In [5]:
for seed in (0, 2):
    Td, td = make_icu_cohort(seed=seed, layout='date')
    fd = ov.tl.audit_clinical_tables(Td, external_reference=False, verbose=False)
    r, p, n = score(fd, td)
    print(f'seed {seed}: recall', {k: round(v, 2) for k, v in r.items()}, f'| precision {p:.2f} ({n} rows)')

seed 0: recall {'after_death': 1.0, 'demographic_change': 1.0, 'age_shift': 1.0, 'height_unit': 1.0, 'weight_lb': 1.0, 'duplicate': 1.0} | precision 1.00 (18 rows)
seed 2: recall {'after_death': 1.0, 'demographic_change': 1.0, 'age_shift': 1.0, 'height_unit': 1.0, 'weight_lb': 1.0, 'duplicate': 1.0} | precision 0.95 (19 rows)


## 5. Full screen by default, targeted on request

On real extracts the graded rules (corpus extremes, jumps in a patient's history) fire on many *correct* rows: heavy-tailed analytes, volatile patients. `mode='targeted'` therefore reports every `high` finding and, per table and graded rule, only the most severe `max(3, screen_rate * n_rows)` `screen` findings (share-of-rows thresholds as in the OHDSI Data Quality Dashboard; alert volume drives alert fatigue). `mode='screen'` returns every rule hit (the recall-first list).

Below, a heavy-tailed analyte is added to the lab table: the screen list grows with correct extremes, the targeted list does not.

In [6]:
Th, th = make_icu_cohort(seed=1, layout='offset')
L = Th['lab']
cr = L['lab_name'] == 'cr'
# heavy tail as a deterministic function of the old value (injected copies stay copies)
u = (pd.to_numeric(L.loc[cr, 'lab_result']) * 1000 * 0.6180339887) % 1
L.loc[cr, 'lab_result'] = np.round(np.exp(0.6 * np.tan(np.pi * (u.clip(0.01, 0.99) - 0.5))), 2).astype(str)
for mode in ('targeted', 'screen'):
    g = ov.tl.audit_clinical_tables(Th, external_reference=False, verbose=False, mode=mode)
    r2, p2, n2 = score(g, th)
    print(f'{mode:8s} recall', {k: round(v, 2) for k, v in r2.items()}, f'| precision {p2:.2f} ({n2} rows)')
print(g.groupby(['rule', 'confidence']).size().unstack(fill_value=0).to_string())

targeted recall {'after_death': 1.0, 'demographic_change': 1.0, 'age_shift': 1.0, 'height_unit': 1.0, 'weight_lb': 1.0, 'duplicate': 1.0} | precision 0.77 (26 rows)
screen   recall {'after_death': 1.0, 'demographic_change': 1.0, 'age_shift': 1.0, 'height_unit': 1.0, 'weight_lb': 1.0, 'duplicate': 1.0} | precision 0.12 (170 rows)
confidence       high  screen
rule                         
after_death         3       0
exact_duplicate     5       0
item_extreme        0       2
outside_stay        3       0
person_constant     6       0
person_numeric      6       3
subject_jump        0     146
unit_factor        12       0


## Summary

| step | call |
|---|---|
| schema (ids, offsets, death encodings) | `ov.tl.infer_clinical_schema(tables)` |
| audit, full screening list (default; recall first) | `ov.tl.audit_clinical_tables(tables)` |
| targeted list (when a precision floor binds) | `mode='targeted'` (keeps every `confidence == 'high'` finding) |
| trim graded rules by share of rows | `max_rule_rate=` |

Typed tables are accepted as they are (ids read as int in one table and float in another are matched; `72.0` equals `72` when looking for duplicate rows).

References: Kahn MG et al. (2016) A harmonized data quality assessment terminology and framework for the secondary use of EHR data. *eGEMs* 4(1):1244. Blacketer C et al. (2021) Increasing trust in real-world evidence through evaluation of observational data quality. *JAMIA* 28(10):2251 (OHDSI DataQualityDashboard). Ancker JS et al. (2017) Effects of workload, work complexity, and repeated alerts on alert fatigue in a clinical decision support system. *BMC Med Inform Decis Mak* 17:36. Estiri H et al. (2019) A clustering approach for detecting implausible observation values in electronic health records data. *BMC Med Inform Decis Mak* 19:142.